# Braitenberg Vehicle — Duck Avoidance

In this task your robot will navigate autonomously through a field of yellow duckies and avoid them.
You will implement two functions:

1. **`detect_ducks(frame, lower_hsv, upper_hsv)`** — finds duck pixels using HSV colour thresholding
2. **`get_wheel_speeds(left_signal, right_signal, gain, const)`** — maps detection signals to wheel commands

Use this notebook to understand the concepts and test your implementation on sample images before running it on the robot or simulation.

---

## 1 · What is a Braitenberg Vehicle?

A **Braitenberg vehicle** is a thought experiment from Valentino Braitenberg's 1984 book *Vehicles*.
It is the simplest possible "nervous system": sensors connect directly to motors, with no planning or memory.
Despite the simplicity, the resulting behaviour can look surprisingly intelligent.

### Vehicle 2b — Fear (avoidance)

```
      LEFT sensor ──────────────────► RIGHT motor
      RIGHT sensor ─────────────────► LEFT motor
```

Each sensor is **cross-wired** to the *opposite* motor with an **excitatory** (positive) connection.

| Situation | Effect |
|-----------|--------|
| Duck on the **left** | Left signal high → Right wheel speeds up → robot turns **left** (away from duck) |
| Duck on the **right** | Right signal high → Left wheel speeds up → robot turns **right** (away from duck) |
| No ducks | Both signals low → both wheels at baseline `const` → robot drives **straight** |

The formula is:

```
left_wheel  = const + gain × right_signal
right_wheel = const + gain × left_signal
```

---

## 2 · How the sensors work

We have a single forward-facing camera. We split the image down the middle:

```
┌────────────┬────────────┐
│  LEFT half │ RIGHT half │
│  (sensor L)│  (sensor R)│
└────────────┴────────────┘
```

For each half we count how many pixels match the duck colour. Dividing by the total pixels gives a signal in **[0, 1]**.

## 3 · Setup — imports and sample images

In [ ]:
import os, sys
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Add project root to path so we can import our packages
notebook_dir = os.path.abspath('')
project_root = os.path.normpath(os.path.join(notebook_dir, '..', '..', '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Paths to sample images
SAMPLES_DIR = os.path.join(project_root, 'tasks', 'assets', 'samples', 'big-duck')
sample_files = sorted([
    os.path.join(SAMPLES_DIR, f)
    for f in os.listdir(SAMPLES_DIR)
    if f.lower().endswith(('.jpg', '.png'))
])
print(f'Found {len(sample_files)} sample images in {SAMPLES_DIR}')

def load_rgb(path):
    """Load an image as RGB numpy array (same format as the camera driver)."""
    bgr = cv2.imread(path)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def show(img, title='', cmap=None):
    plt.figure(figsize=(8, 4))
    plt.imshow(img, cmap=cmap)
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Preview the first sample image
sample = load_rgb(sample_files[0])
show(sample, f'Sample image — shape {sample.shape}')

## 4 · HSV colour space

RGB is hard to threshold by colour (lighting changes affect all three channels equally).
**HSV** (Hue, Saturation, Value) separates colour (H) from brightness (V), making thresholding robust.

OpenCV convention:
* **H** — 0 to 179 (note: half of 360°)
* **S** — 0 to 255 (0 = grey, 255 = fully saturated)
* **V** — 0 to 255 (0 = black, 255 = bright)

Yellow duckies are roughly: **H = 15–35**, **S = 100–255**, **V = 100–255**.

The cell below lets you experiment with the thresholds interactively.

In [ ]:
# ── Try different HSV thresholds here ────────────────────────────
LOWER_HSV = np.array([15, 100, 100])   # [H, S, V]
UPPER_HSV = np.array([35, 255, 255])
# ─────────────────────────────────────────────────────────────────

frame = load_rgb(sample_files[0])

# Convert RGB → BGR → HSV (OpenCV works in BGR)
bgr   = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
hsv   = cv2.cvtColor(bgr,   cv2.COLOR_BGR2HSV)

# Binary mask: white where pixel matches duck colour
mask  = cv2.inRange(hsv, LOWER_HSV, UPPER_HSV)

# Green overlay for visualisation
overlay = frame.copy()
overlay[mask > 0] = [0, 200, 0]
blended = cv2.addWeighted(frame, 0.6, overlay, 0.4, 0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(frame);   axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(mask, cmap='gray'); axes[1].set_title('Mask (white = duck)'); axes[1].axis('off')
axes[2].imshow(blended); axes[2].set_title('Overlay'); axes[2].axis('off')
plt.tight_layout(); plt.show()

pct = 100 * mask.sum() / 255 / mask.size
print(f'Duck pixels: {mask.sum()//255} ({pct:.1f}% of frame)')

## 5 · Splitting the frame into left / right halves

In [ ]:
frame = load_rgb(sample_files[0])
h, w  = frame.shape[:2]
half  = w // 2

left_half  = frame[:, :half]
right_half = frame[:, half:]

# Draw split line on a copy
vis = frame.copy()
cv2.line(vis, (half, 0), (half, h), (255, 0, 0), 3)
cv2.putText(vis, 'LEFT sensor', (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 50, 50), 2)
cv2.putText(vis, 'RIGHT sensor', (half + 10, 30),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (50, 50, 255), 2)

show(vis, 'Left / right split')

## 6 · Your task — implement `detect_ducks()`

Open the file and fill in the body:

```
tasks/braitenberg/packages/color_detection.py
```

The function must:
1. Convert `frame_rgb` → BGR → HSV
2. Create a binary mask with `cv2.inRange`
3. Split the mask into left/right halves
4. Compute the **fraction** of white pixels in each half (duck count ÷ total pixels)
5. Return `(left_signal, right_signal)`

Test it in the cell below.

In [ ]:
from tasks.braitenberg.packages import color_detection
import importlib
importlib.reload(color_detection)  # picks up edits without restarting kernel

LOWER = np.array([15, 100, 100])
UPPER = np.array([35, 255, 255])

results = []
for path in sample_files[:6]:
    frame = load_rgb(path)
    left_sig, right_sig = color_detection.detect_ducks(frame, LOWER, UPPER)
    results.append((os.path.basename(path), left_sig, right_sig))
    print(f'{os.path.basename(path):20s}  left={left_sig:.4f}  right={right_sig:.4f}')

In [ ]:
# Visual check: overlay detection on first 3 samples
fig, axes = plt.subplots(3, 2, figsize=(12, 9))
for row, path in enumerate(sample_files[:3]):
    frame  = load_rgb(path)
    ls, rs = color_detection.detect_ducks(frame, LOWER, UPPER)

    # Rebuild mask for display
    bgr  = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
    hsv  = cv2.cvtColor(bgr,   cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, LOWER, UPPER)
    overlay = frame.copy(); overlay[mask > 0] = [0, 200, 0]
    blended = cv2.addWeighted(frame, 0.6, overlay, 0.4, 0)
    cx = frame.shape[1] // 2
    cv2.line(blended, (cx, 0), (cx, frame.shape[0]), (255, 0, 0), 2)

    axes[row, 0].imshow(blended); axes[row, 0].axis('off')
    axes[row, 0].set_title(f'{os.path.basename(path)}')
    axes[row, 1].bar(['Left', 'Right'], [ls, rs], color=['#1f77b4', '#ff7f0e'])
    axes[row, 1].set_ylim(0, 1); axes[row, 1].set_ylabel('Signal')
    axes[row, 1].set_title(f'L={ls:.3f}  R={rs:.3f}')

plt.tight_layout(); plt.show()

## 7 · Your task — implement `get_wheel_speeds()`

Open the file:

```
tasks/braitenberg/packages/braitenberg_vehicle.py
```

Implement the **Vehicle 2b avoidance** formula:

```
left_wheel  = const + gain × right_signal
right_wheel = const + gain × left_signal
```

Then clamp both results to **[-1.0, 1.0]** and return `(left_speed, right_speed)`.

In [ ]:
from tasks.braitenberg.packages import braitenberg_vehicle
importlib.reload(braitenberg_vehicle)

GAIN  = 0.7
CONST = 0.3

# Unit tests — verify your formula
test_cases = [
    # (left_sig, right_sig, expected_left, expected_right, description)
    (0.0, 0.0, CONST, CONST, 'No ducks → straight'),
    (0.5, 0.0, CONST + GAIN * 0.0, CONST + GAIN * 0.5, 'Duck on left → right wheel faster'),
    (0.0, 0.5, CONST + GAIN * 0.5, CONST + GAIN * 0.0, 'Duck on right → left wheel faster'),
    (1.0, 1.0, min(1.0, CONST + GAIN), min(1.0, CONST + GAIN), 'Ducks everywhere → both clamped'),
]

all_ok = True
for ls, rs, exp_l, exp_r, desc in test_cases:
    got_l, got_r = braitenberg_vehicle.get_wheel_speeds(ls, rs, GAIN, CONST)
    ok = abs(got_l - exp_l) < 1e-6 and abs(got_r - exp_r) < 1e-6
    status = '✓' if ok else '✗'
    print(f'{status} {desc}')
    if not ok:
        print(f'  Expected ({exp_l:.3f}, {exp_r:.3f}), got ({got_l:.3f}, {got_r:.3f})')
        all_ok = False

print('\nAll tests passed!' if all_ok else '\nSome tests failed — check your formula.')

## 8 · End-to-end test on sample images

Run the full pipeline (detect → decide → motors) on several samples.

In [ ]:
importlib.reload(color_detection)
importlib.reload(braitenberg_vehicle)

LOWER = np.array([15, 100, 100])
UPPER = np.array([35, 255, 255])
GAIN  = 0.7
CONST = 0.3

print(f'{'Image':25s}  {'L-sig':>6}  {'R-sig':>6}  {'L-spd':>6}  {'R-spd':>6}  Action')
print('-' * 72)

for path in sample_files:
    frame = load_rgb(path)
    ls, rs = color_detection.detect_ducks(frame, LOWER, UPPER)
    lw, rw = braitenberg_vehicle.get_wheel_speeds(ls, rs, GAIN, CONST)

    if ls > rs + 0.01:
        action = 'turn LEFT  (duck on left)'
    elif rs > ls + 0.01:
        action = 'turn RIGHT (duck on right)'
    else:
        action = 'straight'

    print(f'{os.path.basename(path):25s}  {ls:6.3f}  {rs:6.3f}  {lw:6.3f}  {rw:6.3f}  {action}')

## 9 · Running in simulation

Once your functions pass the tests above, launch the Godot simulation from the project root:

```bash
python launch.py --sim --task braitenberg
```

Open the URL printed in the terminal. The dashboard shows:
* **Green overlay** — pixels detected as ducks
* **Orange bars** at the top — left/right signal strength
* **Sliders** — tune HSV thresholds and gain/const in real time without restarting

### Tuning tips

| Symptom | Fix |
|---------|-----|
| Robot does not react to ducks | Widen HSV range or increase `gain` |
| Robot reacts to non-duck objects | Narrow HSV range (especially H) |
| Robot oscillates wildly | Reduce `gain` |
| Robot too slow / stops | Increase `const` |
| Robot crashes into ducks | Increase `gain`, robot needs to react earlier |

## 10 · Running on the real robot

```bash
python launch.py --run --bot <your_bot_name> --task braitenberg
```

In the TTF session, tune HSV values on the real robot because lighting conditions differ from simulation.
Place several ducks on the floor and watch the robot navigate through them.

> **Do not steal the ducks.**